# Coding AI Fundamentals

Coding models sit at the intersection of language modeling and software engineering. They are not merely “LLMs that sometimes write code”—they are systems trained, prompted, evaluated, and productized around the realities of repositories, tests, diffs, and developer workflows.

This notebook builds a senior-level mental model: what coding models are, why specialization matters, how capability layers stack from autocomplete to agents, and how to reason about quality beyond leaderboard scores.

```mermaid
flowchart LR
  NL[Natural language / IDE context] --> CM[Coding Model]
  CM --> C1[Completion]
  CM --> C2[Generation]
  CM --> C3[Edit / Diff]
  CM --> C4[Explain / Review]
  CM --> C5[Tool-using Agent]
  C5 --> Tools[Tests / Linter / Search / VCS]
```


## Learning Objectives

By the end of this notebook you will be able to:

1. Define coding models and contrast them with general chat LLMs
2. Map the spectrum from inline completion → chat → repo agents → CI bots
3. Enumerate quality dimensions that matter in production engineering
4. Apply core prompt patterns (spec→tests→code, diff-oriented edits)
5. Reason about context windows, repository packing, and retrieval
6. Build a minimal client and a local heuristic evaluator

**Prerequisites:** Python 3.10+, basic HTTP/APIs, familiarity with git and unit tests.


## 1. What Are Coding Models?

### Definition
A **coding model** is a large language model whose pretraining mixture, fine-tuning objectives, tokenizer design, and evaluation suite are optimized for software artifacts—source files, diffs, build logs, issue trackers, and API docs.

### Why it matters
- **Developer leverage:** autocomplete and agents change throughput of feature delivery
- **Risk surface:** bad code ships faster; security and maintainability matter more
- **Product design:** latency budgets for ghost-text differ radically from multi-minute agents
- **Cost:** repository-scale context and tool loops dominate spend

### How it works (intuition)
Think of three training layers:
1. **Pretrain on code + text** — learn syntax, APIs, idioms, and natural language about code
2. **Specialized objectives** — Fill-in-the-Middle (FIM), next-edit prediction, instruction following on coding tasks
3. **Post-train / RL** — preference data, unit-test rewards, tool-use trajectories

### Core capabilities
| Capability | User-visible form | Typical latency |
|------------|-------------------|-----------------|
| Completion / FIM | Ghost text | <200ms–1s |
| NL → code | Chat / generate | 1–30s |
| Repair / debug | Fix failing tests | seconds–minutes |
| Refactor / migrate | Multi-file edits | minutes |
| Repo Q&A | Chat over index | seconds |
| Agent loops | Plan→act→observe | minutes–hours |

### Pitfalls
- Treating HumanEval as a proxy for “good at my monorepo”
- Ignoring security: models happily invent insecure patterns
- Over-trusting long context without retrieval or structure
- Confusing **chat fluency** with **executable correctness**

### When to use a specialized coding model
Use coding-specialized models when you need FIM latency, strong repo edits, or code-bench performance. General models can still excel at architecture discussion and high-level design.


## 2. The Coding AI Spectrum

```
┌─────────────────────────────────────────────────────────────────┐
│  CI / PR bots     automated review, fix suggestions, checks     │
├─────────────────────────────────────────────────────────────────┤
│  Repo agents      multi-step plan, tools, long-running tasks    │
├─────────────────────────────────────────────────────────────────┤
│  Chat assistants  conversational edits, explanations            │
├─────────────────────────────────────────────────────────────────┤
│  Inline complete  FIM / next-token ghost text                   │
└─────────────────────────────────────────────────────────────────┘
         ↑ lower latency, narrower task          higher autonomy ↑
```

| Layer | Examples | Interaction | Context needs |
|-------|----------|-------------|----------------|
| Inline completion | Copilot, Continue, TabNine | Ghost text | Current file + locals |
| Chat assistant | Cursor Chat, IDE Claude | Conversational | Selection + open files |
| Repo agent | SWE-agent style, Devin-like | Multi-step | Search + tests + tree |
| CI agent | PR bots | Automated | Diff + CI logs |

**Product rule:** pick the *thinnest* layer that solves the job. Autocomplete should not spawn an agent; a migration should not rely on single-shot chat.


In [ ]:
# Demo 1 — Capability layer classifier (educational heuristic)
from dataclasses import dataclass
from typing import Literal

Layer = Literal["inline", "chat", "agent", "ci_bot"]

@dataclass
class TaskSpec:
    name: str
    needs_tools: bool
    multi_file: bool
    latency_budget_ms: int
    autonomous: bool

def recommend_layer(t: TaskSpec) -> Layer:
    if t.latency_budget_ms < 800 and not t.needs_tools:
        return "inline"
    if t.autonomous or (t.needs_tools and t.multi_file):
        return "agent"
    if "pr" in t.name.lower() or "review" in t.name.lower():
        return "ci_bot"
    return "chat"

examples = [
    TaskSpec("complete import", False, False, 150, False),
    TaskSpec("explain this function", False, False, 5000, False),
    TaskSpec("migrate auth module", True, True, 600_000, True),
    TaskSpec("pr security review", True, True, 120_000, True),
]
for e in examples:
    print(f"{e.name:28} -> {recommend_layer(e)}")


## 3. Quality Dimensions for Generated Code

Leaderboards measure slices of quality. Production cares about a broader scorecard:

1. **Correctness** — passes tests / matches acceptance criteria
2. **Idiomatic style** — matches language + project conventions
3. **Security** — injection, authz, secrets, unsafe deserialization
4. **Maintainability** — structure, names, coupling
5. **Efficiency** — algorithmic complexity, allocations, N+1 queries
6. **Testability** — seams for mocks, pure cores
7. **Operability** — logs, metrics, error taxonomy
8. **Licensing / provenance** — especially with copy-heavy completions

### Intuition
A model can “look right” (fluent API names) while being wrong (hallucinated methods) or unsafe (`eval`, string-built SQL). Always bind quality to **executable oracles** (tests, typecheckers, scanners) when possible.


In [ ]:
# Demo 2 — Local heuristic evaluator (no API needed)
import ast
from dataclasses import dataclass

@dataclass
class CodeReview:
    parses: bool
    has_eval: bool
    has_exec: bool
    bare_except: bool
    long_functions: list[str]
    score: float
    notes: list[str]

def heuristic_review(src: str) -> CodeReview:
    notes = []
    try:
        tree = ast.parse(src)
        parses = True
    except SyntaxError as e:
        return CodeReview(False, False, False, False, [], 0.0, [f"SyntaxError: {e}"])

    has_eval = any(isinstance(n, ast.Call) and getattr(n.func, "id", None) == "eval" for n in ast.walk(tree))
    has_exec = any(isinstance(n, ast.Call) and getattr(n.func, "id", None) == "exec" for n in ast.walk(tree))
    bare_except = any(isinstance(n, ast.ExceptHandler) and n.type is None for n in ast.walk(tree))
    long_fns = []
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):
            span = (n.end_lineno or n.lineno) - n.lineno + 1
            if span > 40:
                long_fns.append(n.name)

    score = 1.0
    if has_eval or has_exec:
        score -= 0.4
        notes.append("Dangerous dynamic execution")
    if bare_except:
        score -= 0.2
        notes.append("Bare except hides failures")
    if long_fns:
        score -= 0.1
        notes.append(f"Long functions: {long_fns}")
    if not notes:
        notes.append("No heuristic issues found")
    return CodeReview(parses, has_eval, has_exec, bare_except, long_fns, max(score, 0.0), notes)

sample_bad = '''
def run(user_code):
    try:
        return eval(user_code)
    except:
        pass
'''
print(heuristic_review(sample_bad))


## 4. Prompt Patterns for Coding Tasks

### Pattern A — Instruction + constraints
State language, framework, I/O contract, performance bounds, and **non-goals**.

### Pattern B — Spec → tests → code
Ask for tests (or a checklist) before or with implementation to reduce hallucinated APIs.

### Pattern C — Diff-oriented edits
Prefer unified diffs / structured patches over full-file rewrites for large files.

### Pattern D — Evidence-bound answers
Require citations to file paths and line ranges when explaining a repo.

### Pattern E — Fail closed
If context is insufficient, the model should ask or abstain rather than invent modules.


In [ ]:
# Demo 3 — Minimal coding-model client skeleton (placeholder API key)
import os
from typing import Any

# Placeholder — never commit real keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-YOUR_OPENAI_API_KEY_HERE")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

SYSTEM = """You are a senior software engineer.
Prefer correct, secure, idiomatic code.
If requirements are ambiguous, list assumptions explicitly.
Return code in fenced blocks with language tags."""

def build_messages(task: str, context: str = "") -> list[dict[str, str]]:
    user = task if not context else f"Context:\n{context}\n\nTask:\n{task}"
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
    ]

def fake_complete(messages: list[dict[str, str]]) -> dict[str, Any]:
    """Stand-in for an HTTP call — swap for openai/httpx in real use."""
    return {
        "model": "coding-model-placeholder",
        "choices": [{"message": {"role": "assistant", "content": "# TODO: call API\npass\n"}}],
        "usage": {"prompt_tokens": 120, "completion_tokens": 20},
    }

msgs = build_messages(
    "Write a pure function is_anagram(a: str, b: str) -> bool",
    context="Python 3.11, no third-party deps, Unicode-aware",
)
print(fake_complete(msgs))
print("Using key prefix:", OPENAI_API_KEY[:7], "...")


In [ ]:
# Demo 4 — Spec → tests → code prompt assembler
from textwrap import dedent

def assemble_tdd_prompt(spec: str, language: str = "python") -> str:
    return dedent(f"""
    You will solve a coding task in three phases.

    ## Spec
    {spec}

    ## Phase 1 — Acceptance tests
    Write {language} unit tests that encode the spec. Do not implement yet.

    ## Phase 2 — Implementation
    Implement the minimal correct solution.

    ## Phase 3 — Self-review
    List edge cases, complexity, and security notes.
    """).strip()

print(assemble_tdd_prompt("normalize phone numbers to E.164 when country=US"))


## 5. Context Windows and Repository Awareness

### Definition
**Repository awareness** is the ability to condition generations on the right files, symbols, and project conventions—not merely stuffing random files into a long context.

### How it works
Typical stacks combine:
- IDE signals (open tabs, cursor, recent edits)
- Symbol / AST indexes
- Embedding or BM25 retrieval over chunks
- Agent tools (`grep`, `find`, test runner)
- Packing policies (recency, path priors, dependency graphs)

```
User task
   │
   ├─► retrieve candidate chunks
   ├─► rank / rerank
   ├─► pack under token budget
   └─► generate (optionally with tools)
```

### Pitfalls
- **Lost-in-the-middle** for huge packed prompts
- Retrieving similar but wrong APIs (name collisions)
- Ignoring build/test configuration files that define truth


In [ ]:
# Demo 5 — Token budget packer for repo snippets
from dataclasses import dataclass

@dataclass
class Snippet:
    path: str
    text: str
    score: float

def estimate_tokens(text: str) -> int:
    # rough heuristic ~4 chars/token for code-ish English
    return max(1, len(text) // 4)

def pack(snippets: list[Snippet], budget: int) -> list[Snippet]:
    chosen = []
    used = 0
    for s in sorted(snippets, key=lambda x: -x.score):
        cost = estimate_tokens(s.text) + estimate_tokens(s.path) + 8
        if used + cost > budget:
            continue
        chosen.append(s)
        used += cost
    return chosen

snips = [
    Snippet("app/auth.py", "def login():\n    ...\n" * 20, 0.9),
    Snippet("app/utils.py", "def helper():\n    return 1\n", 0.4),
    Snippet("tests/test_auth.py", "def test_login():\n    assert True\n" * 10, 0.8),
]
packed = pack(snips, budget=200)
print([(p.path, estimate_tokens(p.text)) for p in packed])


### Try it yourself — Fundamentals

- Pick a real task from your job and classify it into inline/chat/agent/ci_bot
- Write a quality scorecard (8 dimensions) for evaluating a coding assistant at your company
- Swap `fake_complete` for a real OpenAI-compatible call using env placeholders
- Run `heuristic_review` on three model outputs and compare to human judgment


## Glossary / Key Terms

| Term | Meaning |
|------|--------|
| `FIM` | Fill-in-the-Middle — generate a span given prefix and suffix |
| `Ghost text` | Inline gray suggestion in the editor |
| `SWE-bench` | Benchmark of real GitHub issue→PR resolution tasks |
| `Pass@k` | Probability at least one of k samples passes tests |
| `Repository packing` | Selecting/ordering files/chunks into a prompt budget |
| `Diff-oriented edit` | Model returns a patch rather than a full file |


## Interview Prep — Sample Q&A

Practice answering out loud, then compare to the sample answers.


### Q1. How do coding models differ from general LLMs?

**Sample answer**

Training mix and objectives emphasize code, FIM/edit prediction, and engineering evals. Product latency and tool loops differ. General models may still win at design discussion; coding models often win at completion and repair.


### Q2. When would you choose an agent over chat?

**Sample answer**

When the task needs multi-step tool use, repo navigation, and iterative test feedback—and the latency/cost budget allows. Prefer chat for scoped edits with clear context.


## Deep Dive Workshop — 01 Coding Ai Fundamentals

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `01-coding-ai-fundamentals`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 01-coding-ai-fundamentals)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Coding-models advanced notes

**Fill-in-the-middle formats** differ by vendor; always keep an adapter layer.  
**Repo agents** should treat tests as the north star and protect test files by default.  
**Coding RAG** should combine symbol lookup + BM25 + embeddings; embeddings alone miss identifiers.

| Task | Prefer | Avoid |
|------|--------|-------|
| Ghost text | FIM-capable small/fast model | Giant chat model sync |
| API migration | Agent + tests | Single-shot whole-repo rewrite |
| Explain legacy | Chat + citations | Uncited summaries |


In [ ]:
# Extra demo — diff extraction toy for coding assistants
import re

def extract_fenced_blocks(text: str) -> list[tuple[str, str]]:
    pat = re.compile(r"```(\w+)?\n(.*?)```", re.S)
    return [(m.group(1) or 'txt', m.group(2)) for m in pat.finditer(text)]

sample = '''Here is a fix:\n```python\ndef add(a,b):\n    return a+b\n```\n'''
print(extract_fenced_blocks(sample))


In [ ]:
# Extra demo — simple symbol index for coding RAG
import ast
from collections import defaultdict

def index_symbols(source: str, path: str) -> dict[str, list[str]]:
    tree = ast.parse(source)
    idx = defaultdict(list)
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            idx[n.name].append(f"{path}:{n.lineno}")
    return dict(idx)

print(index_symbols('class Foo:\n  def bar(self):\n    pass\n', 'a.py'))


### Sample interview Q&A — coding models

**Q:** Copilot-quality inline completion is slow. What do you do?  
**A:** Separate completion model from chat model; shrink context to locals + imports; consider speculative decoding / smaller quantized model; measure acceptance rate not just tok/s.

**Q:** How do you evaluate a coding assistant for a monorepo?  
**A:** Private suite: completion acceptance, unit-test pass on generated patches, security scanner findings, and human review on a stratified sample of PR diffs.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `01-coding-ai-fundamentals`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Coding models are specialized LLMs for software engineering workflows
- Match capability layer (inline→agent) to task autonomy and latency
- Quality is multi-dimensional; bind to tests and scanners
- Prompt with constraints, tests-first, and diffs for large edits
- Repository awareness ≠ raw long context—retrieve, rank, pack
